# Vlnová rovnice na hyperboloidálních řezech: přeškálování $\chi = \sqrt{1+R^2}$

Stejný postup jako v `HypSlc_WESystemNumSol.ipynb`, ale místo proměnné $R\psi$ evolvujeme

$$
\tilde\psi \equiv \chi\,\psi, \qquad \chi(R) = \sqrt{1 + R^2},
$$

po vzoru Gautam, Vaño-Viñuales, Hilditch, Bose, PRD **103**, 084045 (2021), rov. (15) a (39):
$\chi \simeq R$ pro velká $R$ (zruší $1/R$ úpadek u $\mathcal{I}^+$), ale $\chi \simeq 1$ u počátku —
takže $\tilde\psi$ je **sudá** a regulární v $r = 0$ jako původní $\psi$ (na rozdíl od liché $R\psi$).

**Rovnice pro $\tilde\psi$ ve fyzikálních souřadnicích.** Dosazením $\psi = \tilde\psi/\chi$ do
$\partial_T^2\psi = \partial_R^2\psi + \frac{2}{R}\partial_R\psi$ dostaneme

$$
\partial_T^2\tilde\psi = \partial_R^2\tilde\psi + a(R)\,\partial_R\tilde\psi + b(R)\,\tilde\psi,
\qquad
a = \frac{2}{R\,\chi^2}, \qquad b = -\frac{3}{\chi^4}
$$

(kontrola: $\psi = 1$, tj. $\tilde\psi = \chi$, dává $\chi'' + a\chi' + b\chi = 0$ ✓).
Člen $a$ má u počátku strukturu $2/R$ jako původní rovnice — ošetří se l'Hospitalem;
u $\mathcal{I}^+$ padá jako $1/R^3$ a $b$ jako $1/R^4$.

**Transformace** (stejná jako dřív): $t = T - h(R)$, $h = \sqrt{S^2+R^2} - S$; $R = r/\Omega$,
$\Omega = 1 - r^2/S^2$; $Q = \sqrt{S^2\Omega^2 + r^2}$, $A = r/Q$, $L^{-1} = \Omega^2/(1+r^2/S^2)$,
$K = L(1-A^2) = (1+r^2/S^2)S^2/Q^2$.

**Systém v proměnných** $\tilde\pi = -\partial_t\tilde\psi$, $\tilde\phi = \partial_r\tilde\psi$:

$$
\partial_t \tilde\psi = -\tilde\pi,
$$
$$
\partial_t \tilde\phi = -\partial_r \tilde\pi + \gamma_2 (\partial_r \tilde\psi - \tilde\phi),
$$
$$
\partial_t \tilde\pi = -\frac{1}{K}\Big[\, 2A\,\partial_r \tilde\pi + A'\tilde\pi
+ L^{-1}\partial_r \tilde\phi + \big(L^{-1}\big)'\tilde\phi
+ (aLA)\,\tilde\pi + a\,\tilde\phi + (bL)\,\tilde\psi \,\Big],
$$

kde nové koeficienty mají v $r$ uzavřenou formu (s $D \equiv \Omega^2 + r^2 = \chi^2\Omega^2$):

$$
aLA = \frac{2\,\Omega\,(1+r^2/S^2)}{Q\,D}, \qquad
a = \frac{2\,\Omega^3}{r\,D}, \qquad
bL = -\frac{3\,(1+r^2/S^2)\,\Omega^2}{D^2}.
$$

Na $\mathcal{I}^+$ všechny tři vymizí ($\Omega = 0$) a rovnice degeneruje na čistou advekci ven,
$\partial_t\tilde\pi = -\partial_r\tilde\pi$ — **žádná okrajová podmínka** (jako dřív, $c_- = 0$).

**Počátek** ($A=0$, $L=K=1$, $A' = 1/S$, $aLA = 2/S$, $bL = -3$): $\tilde\phi$ je lichá, tedy
$\partial_t\tilde\phi(0) = 0$, a pro $\tilde\pi$ dá l'Hospital na členu $a\tilde\phi \to 2\,\partial_r\tilde\phi$

$$
\partial_t\tilde\pi\big|_{r=0} = -3\,\partial_r\tilde\phi - \tfrac{3}{S}\,\tilde\pi + 3\,\tilde\psi .
$$

**Počáteční data** (čistě odcházející): $\psi = f(T-R)/R$ dává
$\partial_T\tilde\psi = -\partial_R\tilde\psi - \tfrac{a}{2}\tilde\psi$, tedy

$$
\tilde\pi = \frac{1+A}{K}\left(\tilde\phi + \frac{aL}{2}\,\tilde\psi\right),
\qquad aL = \frac{2\,\Omega\,(1+r^2/S^2)}{r\,D}.
$$

In [ ]:
import numpy as np

# RK integrátor a derivace

In [ ]:
RK_COEFFICIENT_TABLES = {
    1: ([0], [[0]], [1]),
    2: ([0, 1 / 2], [[0, 0], [1 / 2, 0]], [0, 1]),
    3: ([0, 1 / 2, 1], [[0, 0, 0], [1 / 2, 0, 0], [-1, 2, 0]], [1 / 6, 2 / 3, 1 / 6]),
    4: (
        [0, 1 / 2, 1 / 2, 1],
        [[0, 0, 0, 0], [1 / 2, 0, 0, 0], [0, 1 / 2, 0, 0], [0, 0, 1, 0]],
        [1 / 6, 1 / 3, 1 / 3, 1 / 6],
    ),
    5: (
        [0, 1 / 5, 3 / 10, 4 / 5, 8 / 9, 1, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 5, 0, 0, 0, 0, 0, 0],
            [3 / 40, 9 / 40, 0, 0, 0, 0, 0],
            [44 / 45, -56 / 15, 32 / 9, 0, 0, 0, 0],
            [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729, 0, 0, 0],
            [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656, 0, 0],
            [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
        ],
        [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
    ),
    6: (
        [0, 1 / 3, 2 / 3, 1 / 3, 1 / 2, 1 / 2, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 3, 0, 0, 0, 0, 0, 0],
            [0, 2 / 3, 0, 0, 0, 0, 0],
            [1 / 12, 1 / 3, -1 / 12, 0, 0, 0, 0],
            [-1 / 16, 9 / 8, -3 / 16, -3 / 8, 0, 0, 0],
            [0, 9 / 8, -3 / 8, -3 / 4, 1 / 2, 0, 0],
            [9 / 44, -9 / 11, 63 / 44, 18 / 11, 0, -16 / 11, 0],
        ],
        [11 / 120, 0, 27 / 40, 27 / 40, -4 / 15, -4 / 15, 11 / 120],
    ),
}


In [ ]:
class RKp:
    def __init__(self, order: int = 4):
        """
        Args:
            t0 (float): Starting time
            h (float): Time step
            dydt (function): function in shape  dy/dt = f(t, y)
            tmax (float, optional): Stopping time of integration. Defaults to None.
            order (int, optional): Order of RK method used. Currently implemented are {1,2,3,4,5}. Defaults to 4.
        """

        self.order = order

        # load RK coefficients
        (c, A, b) = RK_COEFFICIENT_TABLES[order]
        (self.c, self.A, self.b) = (np.array(c), np.array(A), np.array(b))

    # initialization of values before integration process begins
    def Initialize(self, y0: np.ndarray, dydt):
        if len(y0) % 2 == 1:
            raise Exception("Wrong length of initial vector")
        self.y = y0.copy()
        self.ICS = y0.copy()
        self.y_history = [y0.copy()]
        self.dydt = dydt

    # integrate the given system
    def Integrate(self, t0: float, h: float, tmax: float = None):
        self.t0 = t0
        self.t = t0
        self.h = h

        while True:
            if self.t + h < tmax:
                self.NextStep()
            else:
                if tmax - self.t > 1e-14:
                    self.h = tmax - self.t
                    self.NextStep()
                break

    # calculate ks for general Butcher table
    def _getKs(self):
        k = np.zeros((len(self.c),) + np.shape(self.y))
        for i in range(len(self.c)):
            t = self.t + self.h * self.c[i]

            dy = np.zeros_like(self.y)
            for j in range(i):
                dy += self.A[i, j] * k[j]
            y = self.y + self.h * dy

            k[i] = self.dydt(t, y)
        return k.copy()

    # perform one step of RKp method
    def NextStep(self):
        # obtain k values
        k = self._getKs()
        # update y and t values
        self.y += self.h * sum(self.b[i] * k[i] for i in range(len(self.c)))
        self.t += self.h
        # save y value
        self.y_history.append(self.y.copy())
        return (self.t - self.t0) // self.h

    def GetHistory(self):
        return self.y_history

    @classmethod
    def GetAllImplemented(cls) -> dict:
        return RK_COEFFICIENT_TABLES.keys()


In [ ]:
def deriv_r_o4(arr, dx):
    res = np.zeros_like(arr)
    n = len(arr)

    if n < 5:
        raise ValueError("Pole musí mít alespoň 5 bodů pro 4. řád přesnosti.")

    # Central difference
    res[2:-2] = (-arr[4:] + 8 * arr[3:-1] - 8 * arr[1:-3] + arr[:-4]) / (12 * dx)

    # Borders with 4th order treatment
    res[0] = (-25*arr[0] + 48*arr[1] - 36*arr[2] + 16*arr[3] - 3*arr[4]) / (12 * dx)
    res[1] = (-3*arr[0] - 10*arr[1] + 18*arr[2] - 6*arr[3] + arr[4]) / (12 * dx)

    res[-1] = (25*arr[-1] - 48*arr[-2] + 36*arr[-3] - 16*arr[-4] + 3*arr[-5]) / (12 * dx)
    res[-2] = (3*arr[-1] + 10*arr[-2] - 18*arr[-3] + 6*arr[-4] - arr[-5]) / (12 * dx)

    return res

# Koeficienty transformace a přeškálování

In [ ]:
# --- Kompaktifikace, výšková funkce a koeficienty systému ---
S = 1.0    # poloha scri v kompaktifikované souřadnici
N_h = 202  # sudý počet bodů kvůli kontrole v RKp.Initialize
rh = np.linspace(0.0, S, N_h)
dxh = rh[1] - rh[0]

Omega = 1 - (rh / S) ** 2
Q = np.sqrt(S**2 * Omega**2 + rh**2)
A = rh / Q                            # A(0) = 0, A(scri) = 1 přesně
invL = Omega**2 / (1 + (rh / S) ** 2) # 1/L, na scri 0
K = (1 + (rh / S) ** 2) * S**2 / Q**2 # K(0) = 1, K(scri) = 2

# Derivace koeficientů (hladké funkce, stačí numericky)
dA_dr = deriv_r_o4(A, dxh)
dinvL_dr = deriv_r_o4(invL, dxh)

# --- Koeficienty z přeškálování chi = sqrt(1+R^2) ---
D_chi = Omega**2 + rh**2  # = chi^2 Omega^2
aLA = 2 * Omega * (1 + (rh / S) ** 2) / (Q * D_chi)         # aLA(0) = 2/S, na scri 0
a_phi = np.zeros(N_h)                                       # a = 2 Omega^3 / (r D); a[0] přes l'Hospital
a_phi[1:] = 2 * Omega[1:] ** 3 / (rh[1:] * D_chi[1:])
bL = -3 * (1 + (rh / S) ** 2) * Omega**2 / D_chi**2         # bL(0) = -3, na scri 0
aL_half = np.zeros(N_h)                                     # aL/2 pro počáteční data
aL_half[1:] = Omega[1:] * (1 + (rh[1:] / S) ** 2) / (rh[1:] * D_chi[1:])

# --- Charakteristické rychlosti (principální část se přeškálováním nemění) ---
c_plus = (1 + A) / K
c_minus = -(1 - A) / K
print(f"max |c+| = {np.abs(c_plus).max():.3f}, max |c-| = {np.abs(c_minus).max():.3f}")
print(f"kontrola na scri: aLA = {aLA[-1]:.1e}, a = {a_phi[-1]:.1e}, bL = {bL[-1]:.1e}, K = {K[-1]:.3f}")

In [ ]:
def system_rhs_chi(dx, gamma2, S, A, invL, K, dA_dr, dinvL_dr, aLA, a_phi, bL):
  def _system(t, y):
      """
      d tpsi /dt = - tpi
      d tphi /dt = - d tpi/dr + gamma2 (d tpsi/dr - tphi)
      d tpi /dt = - [ 2A d tpi/dr + A' tpi + (1/L) d tphi/dr + (1/L)' tphi
                      + aLA tpi + a tphi + bL tpsi ] / K
      Bez okrajové podmínky na scri (c_- = 0, čistý outflow).
      """
      N = len(y) // 3
      tpsi = y[0:N]
      tphi = y[N : 2 * N]
      tpi = y[2 * N : 3 * N]

      dtpsi_dt = np.zeros(N)
      dtphi_dt = np.zeros(N)
      dtpi_dt = np.zeros(N)

      # --- d tpsi /dt = -tpi ---
      dtpsi_dt[:] = - tpi

      # --- d tphi /dt = - d tpi/dr + gamma2 (d tpsi/dr - tphi) ---
      dtphi_dt[1:] = - deriv_r_o4(tpi, dx)[1:] + gamma2 * (deriv_r_o4(tpsi, dx) - tphi)[1:]
      # Sudost tpsi zaručuje v počátku tphi (= d tpsi/dr) = 0
      dtphi_dt[0] = 0.0

      # --- d tpi /dt ---
      dtpi_dr = deriv_r_o4(tpi, dx)
      dtphi_dr = deriv_r_o4(tphi, dx)
      dtpi_dt[1:] = - (2 * A * dtpi_dr + dA_dr * tpi + invL * dtphi_dr + dinvL_dr * tphi
                       + aLA * tpi + a_phi * tphi + bL * tpsi)[1:] / K[1:]
      # Počátek l'Hospitalem: a*tphi -> 2 d tphi/dr, dohromady s (1/L) d tphi/dr = d tphi/dr
      # a s A' = 1/S, aLA = 2/S, bL = -3, K = 1:
      dtpi_dt[0] = - 3 * dtphi_dr[0] - (3 / S) * tpi[0] + 3 * tpsi[0]

      return np.concatenate([dtpsi_dt, dtphi_dt, dtpi_dt])
  return _system


# Počáteční data a integrace

In [ ]:
# --- Parametry pulsu a integrace ---
rh0 = 0.4       # střed balíku v kompaktifikované souřadnici
sigma_h = 0.05  # šířka balíku
timestep_h = 0.001
tmax_h = 2.0

# --- Počáteční data: čistě odcházející balík ---
tpsi = np.exp(-np.power(rh - rh0, 2) / np.power(sigma_h, 2))
tphi = deriv_r_o4(tpsi, dxh)
tpi = (1 + A) * (tphi + aL_half * tpsi) / K   # podmínka odchodnosti
tpi[0] = 0.0  # balík je v počátku ~ e^{-64}, bezpečně nula

y0_h = np.concatenate([tpsi, tphi, tpi])
print(f"Počáteční data připravena. Celková délka vektoru y0: {len(y0_h)} (3x {N_h})")

solver_h = RKp(order=6)
solver_h.Initialize(y0_h, system_rhs_chi(dxh, 0.1, S, A, invL, K, dA_dr, dinvL_dr, aLA, a_phi, bL))
solver_h.Integrate(t0=0.0, h=timestep_h, tmax=tmax_h)

print(f"Integrace ukončena")

history_chi = np.array(solver_h.GetHistory())[:, 0:N_h]  # pouze tpsi
i_half = int(round(1.0 / timestep_h))
print(f"max|tpsi| v t = 1.0 (po odchodu balíku): {np.abs(history_chi[i_half]).max():.3e}")
print(f"max|tpsi| v t = {tmax_h}: {np.abs(history_chi[-1]).max():.3e}  <- růst nestability v počátku, viz níže")

# Očekávaná amplituda na scri: R psi se zachovává, tpsi = (chi/R) * (R psi);
# chi/R -> 1 na scri, na startu chi/R = sqrt(1+R0^2)/R0
R0_phys = rh0 / (1 - (rh0 / S) ** 2)
print(f"očekávaný poměr amplitudy na scri: {R0_phys / np.sqrt(1 + R0_phys**2):.3f}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# --- ANIMACE ---
every_nth_frame_h = 20

fig, ax = plt.subplots(figsize=(10, 5))
(line,) = ax.plot(rh, history_chi[0, :], lw=2, color="firebrick")

ax.axvline(S, color="black", lw=1.5, linestyle=":")
ax.text(S, 1.15, "$\\mathcal{I}^+$", ha="center", fontsize=12)

ax.set_xlim(0, S)
ax.set_ylim(-1.1, 1.1)
ax.set_title("Hyperboloidální řezy: $\\tilde\\psi(t, r) = \\sqrt{1+R^2}\\,\\psi$")
ax.set_xlabel("$r$ (kompaktifikované)")
ax.set_ylabel("$\\tilde\\psi$")
ax.grid(True, linestyle="--", alpha=0.6)


def update_h(frame):
    line.set_ydata(history_chi[frame * every_nth_frame_h, :])
    return (line,)


ani_h = FuncAnimation(fig, update_h, frames=len(history_chi) // every_nth_frame_h, interval=40, blit=True, cache_frame_data=False)

plt.close()

from IPython.display import HTML
HTML(ani_h.to_jshtml())


In [ ]:
# --- Radiační signál na scri ---
# tpsi na scri = (chi/R -> 1) * R psi, tedy přímo radiační signál jako dřív.
t_h = np.arange(len(history_chi)) * timestep_h
signal = history_chi[:, -1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_h, signal, lw=2, color="firebrick")
ax.set_xlabel("$t$ (hyperboloidální čas)")
ax.set_ylabel("$\\tilde\\psi(t, r_{\\mathcal{I}})$")
ax.set_title("Signál na $\\mathcal{I}^+$")
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()

print(f"max signálu na scri: {np.abs(signal).max():.3f} v čase t = {t_h[np.argmax(np.abs(signal))]:.3f}")

In [ ]:
# --- Diagnostika: růst normy v čase ---
max_trace = np.abs(history_chi).max(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(t_h, max_trace, lw=2, color="firebrick")
ax.set_xlabel("$t$")
ax.set_ylabel("$\\max_r |\\tilde\\psi|$")
ax.set_title("Průchod balíku a následný exponenciální růst módu v počátku")
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()

# hrubý odhad rychlosti růstu z poslední čtvrtiny běhu
i0 = 3 * len(max_trace) // 4
lam = np.polyfit(t_h[i0:], np.log(max_trace[i0:]), 1)[0]
print(f"odhad rychlosti růstu: Re(lambda) ~ {lam:.1f}")

**Poznámka k amplitudě:** na rozdíl od proměnné $R\psi$ (kde amplituda zůstávala přesně 1)
tady balík dorazí na scri s amplitudou $\approx R_0^{\rm fyz}/\sqrt{1+(R_0^{\rm fyz})^2}$ —
zachovává se $R\psi$, zatímco $\tilde\psi = (\chi/R)\,R\psi$ a poměr $\chi/R$ klesne z počáteční
hodnoty k 1 na scri. Výhoda této volby: $\tilde\psi$ je sudá a řádu $\psi$ u počátku, takže parita
i regularita v $r=0$ jsou stejné jako pro nepřeškálované pole (přesně proto ji volí Gautam et al.).

**Poznámka k nestabilitě v počátku:** po odchodu balíku roste v $r = 0$ gridová oscilace
(vlastní číslo semidiskretního operátoru $\mathrm{Re}\,\lambda \approx +24$ na této mřížce,
lokalizované v počátku na $\tilde\pi$). Je to tatáž patologie jednostranných (ne-SBP) stencilů
v kombinaci s l'Hospitalovým ošetřením počátku, kterou má i původní nepřeškálovaný $\psi$-systém
a (se slabším buzením, díky pinům $\chi(0) = \pi(0) = 0$) i varianta s $R\psi$; rychlost růstu
škáluje $\propto 1/\Delta r$. Přesně tohle je motivace pro SBP operátory v Gautam et al. —
sem patří budoucí implementace SBP; Kreiss–Oligerovu disipaci záměrně nepoužíváme.

Varianta s $R\psi$ je v `HypSlc_WESystemNumSol.ipynb`; exaktní vizualizace v `HypSlc_WESolVizualization.ipynb`.